# Phase 4: Train once, save everything (run this FIRST)

**This is the only notebook that trains models.** Notebooks 01-03 load the seed-42 run from `results/`. It trains the CNN baseline and the transformer under 3 training seeds on one fixed split and saves every metric the write-up needs to `results/*.json` (macro/per-class F1, accuracy, kappa, normalised confusion matrix, transition vs. stable F1, per-subject F1, calibration stats). Each (model, seed) is saved the moment it finishes and skipped on re-run, so if Colab disconnects, just re-run the notebook.

**Pipeline v2** (this run replaces the first one): the split is now grouped by *person* (Sleep-EDF has two nights per person; the first run let a person's second night leak into training), and the intended 30-minute crop around the sleep period now actually works (it was a no-op, leaving Wake at ~69% of the test set). Results are stamped with the pipeline version and a result file from an older pipeline raises instead of being reused.

**Before running:** Runtime -> Change runtime type -> **GPU**. Section 0 stops if there is no GPU.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import json

import numpy as np
from scipy import stats
from torch.utils.data import DataLoader

from src.config import BATCH_SIZE, CASSETTE_DIR, LABEL_NAMES, get_device
from src.data import SleepEDFDataset, build_epoch_arrays, discover_subjects, get_subject_split
from src.experiments import RESULTS_DIR, load_all, run_seeds, summarize

SEEDS = [42, 43, 44]
device = get_device()
print(f"device: {device}")

## 0. Setup: GPU check, local copy of the data, results and cache on Drive

Expects the usual Colab setup to be done first (Drive mounted, repo cloned, `pip install -r requirements.txt`). This cell then:
1. stops if there is no GPU;
2. copies `MyDrive/eeg-sleep-data/sleep-cassette` to local disk and checks every file (reading EDFs directly through Drive was flaky: files listed but could not be opened);
3. keeps `results/` on Drive in a **new** folder (`eeg-sleep-results-v2`), so finished runs survive a disconnect and the old v1 results are never mixed in;
4. picks a Drive cache folder for the preprocessed epochs, so the slow EDF loading only happens once.

In [ ]:
import os
import shutil
import subprocess

import torch

from src.config import CASSETTE_DIR, PIPELINE_VERSION, ROOT_DIR

print("pipeline version:", PIPELINE_VERSION)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
ON_COLAB = Path("/content/drive/MyDrive").exists()
if ON_COLAB:
    assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU, then re-run"

if ON_COLAB:
    # 1. data: Drive -> local disk, verified file by file
    drive_src = Path("/content/drive/MyDrive/eeg-sleep-data/sleep-cassette")
    local_dir = Path("/content/local_data/sleep-cassette")
    local_dir.mkdir(parents=True, exist_ok=True)
    if subprocess.run(["rsync", "-a", f"{drive_src}/", f"{local_dir}/"]).returncode != 0:
        shutil.copytree(drive_src, local_dir, dirs_exist_ok=True)

    bad = []
    for src in sorted(drive_src.glob("*.edf")):
        dst = local_dir / src.name
        if not dst.exists() or dst.stat().st_size != src.stat().st_size or dst.stat().st_size == 0:
            bad.append(src.name)
    assert not bad, f"{len(bad)} files missing/truncated after copy, e.g. {bad[:5]}: re-run this cell"

    link = ROOT_DIR / "data"
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    os.symlink("/content/local_data", link)

    # 2. results on Drive (fresh v2 folder). The clone only has the committed v1 archive in results/.
    drive_results = Path("/content/drive/MyDrive/eeg-sleep-results-v2")
    drive_results.mkdir(exist_ok=True)
    results_link = ROOT_DIR / "results"
    if not results_link.is_symlink():
        assert not list(results_link.glob("*.json")), "unexpected result files directly in results/"
        shutil.rmtree(results_link, ignore_errors=True)
        os.symlink(drive_results, results_link)

    CACHE_DIR = Path("/content/drive/MyDrive/eeg-sleep-cache")
else:
    CACHE_DIR = ROOT_DIR / "cache"

n_psg = len(list(CASSETTE_DIR.glob("SC*-PSG.edf")))
n_hyp = len(list(CASSETTE_DIR.glob("SC*-Hypnogram.edf")))
print(f"data: {CASSETTE_DIR} -> {n_psg} PSG / {n_hyp} hypnogram files (expect 153 / 153)")
assert n_psg == n_hyp and n_psg > 0, "data missing or incomplete"
print("results dir:", (ROOT_DIR / "results").resolve())
print("cache dir:  ", CACHE_DIR)

## 1. Build the split (same split as notebooks 01-03)

Grouped by person: both nights of a person are always in the same set. The cell asserts there is no person overlap and that Wake does not dominate the test set.

In [ ]:
from src.data import assert_no_person_overlap, person_id

subjects = discover_subjects(CASSETTE_DIR)
subject_ids = [s.subject_id for s in subjects]
train_ids, val_ids, test_ids = get_subject_split(subject_ids)  # grouped by person
assert_no_person_overlap(train_ids, val_ids, test_ids)
for name, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
    print(f"{name:5s}: {len(ids):3d} recordings from {len({person_id(s) for s in ids}):2d} people")

train_epochs, train_labels, _ = build_epoch_arrays(subjects, train_ids, cache_dir=CACHE_DIR)
val_epochs, val_labels, _ = build_epoch_arrays(subjects, val_ids, cache_dir=CACHE_DIR)
test_epochs, test_labels, test_subject_ids = build_epoch_arrays(subjects, test_ids, cache_dir=CACHE_DIR)

# sanity checks that would have caught both first-run bugs
for name, y in [("train", train_labels), ("val", val_labels), ("test", test_labels)]:
    print(f"{name:5s}: {len(y):6d} epochs, Wake share {100 * np.mean(y == 0):.1f}%")
assert np.mean(test_labels == 0) < 0.5, "Wake dominates the test set: the sleep-period crop is not working"

train_loader = DataLoader(SleepEDFDataset(train_epochs, train_labels), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SleepEDFDataset(val_epochs, val_labels), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(SleepEDFDataset(test_epochs, test_labels), batch_size=BATCH_SIZE, shuffle=False)

## 2. Train every (model, seed)

Resumable: finished runs are skipped.

In [ ]:
run_seeds(train_loader, val_loader, test_loader, test_subject_ids, seeds=SEEDS, device=device)

## 3. Aggregate: mean ± std across seeds

In [ ]:
grouped = load_all()
table = summarize(grouped)
print(table)

## 4. Is the transformer-vs-CNN gap real?

Paired across seeds (same seed index for both models). With only 3 seeds this test has very little power: treat a large p-value as "can't distinguish", not "proven equal". `scripts/person_bootstrap.py` complements it by resampling the test *people* instead.

In [ ]:
by_seed = {name: {r["seed"]: r["macro_f1"] for r in runs} for name, runs in grouped.items()}
common = sorted(set(by_seed["cnn"]) & set(by_seed["transformer"]))
cnn = np.array([by_seed["cnn"][s] for s in common])
tfm = np.array([by_seed["transformer"][s] for s in common])
diff = tfm - cnn

print(f"seeds: {common}")
print(f"CNN         macro F1: {cnn.mean():.4f} +/- {cnn.std(ddof=1):.4f}")
print(f"Transformer macro F1: {tfm.mean():.4f} +/- {tfm.std(ddof=1):.4f}")
print(f"mean diff (transformer - CNN): {diff.mean():+.4f}  (std {diff.std(ddof=1):.4f})")
t, p = stats.ttest_rel(tfm, cnn)
print(f"paired t-test: t={t:.3f}, p={p:.3f}")

print("\nper-class F1 diff (transformer - CNN), mean over seeds:")
for c in LABEL_NAMES:
    d = np.mean([[r["per_class_f1"][c] for r in grouped["transformer"] if r["seed"] == s][0]
                 - [r["per_class_f1"][c] for r in grouped["cnn"] if r["seed"] == s][0] for s in common])
    print(f"  {c:<4}{d:+.4f}")

## 5. Other numbers for `results.md`

Transformer seed-averaged confusion matrix, per-recording spread, and calibration gap.

In [ ]:
tf_runs = grouped["transformer"]
cm_mean = np.mean([r["confusion_normalized"] for r in tf_runs], axis=0)
print("transformer confusion matrix (row-normalised, mean over seeds):")
print("      " + "".join(f"{n:>7}" for n in LABEL_NAMES))
for name, row in zip(LABEL_NAMES, cm_mean):
    print(f"{name:<6}" + "".join(f"{v:>7.2f}" for v in row))

print("\ntop confusion pairs by seed (true, pred, count):")
for r in tf_runs:
    print(f"  seed {r['seed']}: {r['top_confusion_pairs'][:3]}")

subj = np.array([[r["per_subject_f1"][k] for k in sorted(r["per_subject_f1"])] for r in tf_runs]).mean(axis=0)
print(f"\nper-subject macro F1 (seed-mean): min {subj.min():.3f}, median {np.median(subj):.3f}, max {subj.max():.3f}")

print(f"mean confidence, correct:   {np.mean([r['mean_conf_correct'] for r in tf_runs]):.3f}")
print(f"mean confidence, incorrect: {np.mean([r['mean_conf_incorrect'] for r in tf_runs]):.3f}")

## 6. Save everything

`results/` is linked to `MyDrive/eeg-sleep-results-v2`, so everything is already saved on Drive: one `.json` (metrics + training history) and one `_preds.npz` (predictions, labels, probabilities) per model and seed. Download that folder into the repo's `results/` and the rest (figures, `results.md`) can be regenerated locally without a GPU.